In [26]:
import numpy as np
import torch
import librosa
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from collections import Counter

model_dir = f'D:\Intership\model_final'  
processor = WhisperProcessor.from_pretrained(model_dir)
model = WhisperForConditionalGeneration.from_pretrained(model_dir)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

def transcribe_audio(file_path):
    audio, sr = librosa.load(file_path, sr=16000)
    inputs = processor(audio, sampling_rate=16000, return_tensors="pt")
    input_features = inputs.input_features.to(device)

    model.eval()
    with torch.no_grad():
        predicted_ids = model.generate(input_features)

    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    return transcription

reference = "उसने चौदह किताबें खरीदीं"  
audio_paths = [
    r"D:\Intership\audio\238079.wav",
    r"D:\Intership\audio\238123.wav",
    r"D:\Intership\audio\239492.wav",
    r"D:\Intership\audio\240907.wav"
]

models = [transcribe_audio(path) for path in audio_paths]

def normalize_word(word):
    return word.strip()

def tokenize(sentence):
    return [normalize_word(w) for w in sentence.split()]

def align_sequences(ref, hyp):
    n, m = len(ref), len(hyp)
    dp = np.zeros((n + 1, m + 1))

    for i in range(n + 1):
        dp[i][0] = i
    for j in range(m + 1):
        dp[0][j] = j

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = 0 if ref[i - 1] == hyp[j - 1] else 1
            dp[i][j] = min(
                dp[i - 1][j] + 1,  # deletion
                dp[i][j - 1] + 1,  # insertion
                dp[i - 1][j - 1] + cost  # substitution
            )

    aligned_ref = []
    aligned_hyp = []

    i, j = n, m
    while i > 0 or j > 0:
        if i > 0 and dp[i][j] == dp[i - 1][j] + 1:
            aligned_ref.append(ref[i - 1])
            aligned_hyp.append("<del>")
            i -= 1
        elif j > 0 and dp[i][j] == dp[i][j - 1] + 1:
            aligned_ref.append("<ins>")
            aligned_hyp.append(hyp[j - 1])
            j -= 1
        else:
            aligned_ref.append(ref[i - 1])
            aligned_hyp.append(hyp[j - 1])
            i -= 1
            j -= 1

    return aligned_ref[::-1], aligned_hyp[::-1]

def align_all(reference, models):
    ref_tokens = tokenize(reference)
    aligned_sequences = []

    for model in models:
        hyp_tokens = tokenize(model)
        _, aligned_hyp = align_sequences(ref_tokens, hyp_tokens)
        aligned_sequences.append(aligned_hyp)

    return ref_tokens, aligned_sequences

def build_lattice(ref_tokens, aligned_models):
    lattice = []

    for i in range(len(ref_tokens)):
        bin_words = set()
        bin_words.add(ref_tokens[i])  # always include reference word

        for model_seq in aligned_models:
            if i < len(model_seq):
                word = model_seq[i]
                if word not in ["<del>", "<ins>"]:
                    bin_words.add(word)

        lattice.append(list(bin_words))

    return lattice

def lattice_wer(lattice, hypothesis, reference):
    hyp_tokens = tokenize(hypothesis)
    ref_tokens = tokenize(reference)

    S = D = I = 0

    for i in range(len(lattice)):
        ref_word = ref_tokens[i]

        if i >= len(hyp_tokens):
            D += 1
        else:
            pred_word = hyp_tokens[i]

            if pred_word == ref_word:
                continue  # exact match
            elif pred_word in lattice[i]:
                S += 0.5  # partial credit for valid alternatives
            else:
                S += 1  # substitution penalty

    if len(hyp_tokens) > len(lattice):
        I += len(hyp_tokens) - len(lattice)  

    return (S + D + I) / len(lattice)

ref_tokens, aligned_models = align_all(reference, models)

lattice = build_lattice(ref_tokens, aligned_models)

print("\n🔹 Lattice:")
for i, bin_words in enumerate(lattice):
    print(f"Bin {i}: {bin_words}")

print("\n🔹 Lattice-based WER:")
for i, model in enumerate(models):
    wer = lattice_wer(lattice, model, reference)
    print(f"Model {i + 1}: {wer:.3f}")

<>:8: SyntaxWarning: invalid escape sequence '\I'
<>:8: SyntaxWarning: invalid escape sequence '\I'
C:\Users\Dexter\AppData\Local\Temp\ipykernel_5452\1194180144.py:8: SyntaxWarning: invalid escape sequence '\I'
  model_dir = f'D:\Intership\model_final'  # Update to your fine-tuned model directory


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.log


🔹 Lattice:
Bin 0: ['उस', 'तो', 'उसने', 'जी']
Bin 1: ['मैंने', 'बार', 'चौदह']
Bin 2: ['ना', 'किताबें', 'आप']
Bin 3: ['तो', 'परवार', 'खरीदीं']

🔹 Lattice-based WER:
Model 1: 0.875
Model 2: 13.750
Model 3: 0.875
Model 4: 36.250
